# S8 · Mantenimiento predictivo e IA · ¿Cuánto le queda y cuándo intervenir?
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/sesion_08_mantenimiento_predictivo_e_ia/S8_AC_CLASE_Colab_ALUMNO.ipynb)

## Objetivo
Estimar la **vida remanente** (RUL, *remaining useful life*) de una máquina con la tendencia de un **indicador de
salud**, y decidir **cuándo** dar la alarma con la **matriz de confusión** y el **costo** de cada error. Comparar una
regla sencilla (un umbral) con un modelo de **aprendizaje automático** que combina varios sensores, en máquinas que
el modelo no vio. Es el mantenimiento **predictivo**: se interviene antes de la falla, no por calendario ni después.

## La situación
En la sesión 7 la vibración dijo **qué** falla; ahora hay que decir **cuándo**. En clase trabajas con **tu** bomba (su
indicador de salud y su tendencia) y con una flota de bombas vigiladas por una regla de alarma. En la práctica usas
datos de la NASA: 100 motores turbofan simulados que operan desde nuevos hasta fallar por desgaste del compresor, con
21 sensores; tu motor y tu «hoy» dependen de tu código.

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero): la vida remanente de tu bomba, la precisión y la sensibilidad de la alarma de la flota, y su costo. En cada uno escribes tu resultado, la sustitución con tus números y su unidad, y lo que predice | La fecha de tu intervención y si la alarma conviene |
| **B · Práctica** | 80 min | Predices la vida remanente de tu motor con una recta y la comparas con la real y con una curva; eliges un umbral por costo; comparas el umbral con un modelo de IA en 100 motores nuevos | Gráficas firmadas, tu conclusión y el archivo `S8_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google y contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado. ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno ·
   cierre 1 pt. Se califica que el resultado sea correcto para **tus** datos y que la sustitución y la predicción
   sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas llevan **tus** números.

## Cómo se usa este cuaderno · no necesitas saber programar
Sólo cambias lo que está después del signo **=** en los renglones marcados con ✎.

| Para… | Haz esto |
|---|---|
| Ejecutar una celda | Clic en ▶ a su izquierda (o Ctrl + Enter). Ejecuta las celdas en orden, de arriba hacia abajo |
| Escribir un número | Después del **=**, con punto decimal y sin comas de miles: `u_A = 0.0316`, `MEMORIA = 32000`. Usa al menos 3 cifras significativas |
| Escribir un texto | Entre comillas: `SUST_1 = "70 / 2^10 = 0.0684 °C"`. No borres las comillas |
| Calcular | Con calculadora o con tu hoja de cálculo. Si sabes Python puedes escribir la operación en vez del número: `+ − * /`, potencia `**`, raíz `np.sqrt(x)`, eˣ `np.exp(x)` |
| Ver tus datos | Están impresos debajo de la celda «Tus datos»; usa esos números |
| Un mensaje en rojo | Lee la **última línea**: dice qué falta. Corrige y vuelve a ejecutar esa celda |
| Antes de entregar | **Entorno de ejecución → Reiniciar y ejecutar todo**, y revisa que todo salga ✔ |

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    claves = [k.lower() for k in resultados]
    if len(set(claves)) != len(claves):    # la hoja busca sin distinguir mayúsculas: U y u serían el mismo dato
        raise ValueError("Dos resultados tienen el mismo nombre salvo mayúsculas; renómbrelos.")
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    # datos: lista de números (se nombran por su valor) o {descripción: valor} (se nombran sin revelar el valor)
    pares = datos.items() if isinstance(datos, dict) else [(None, d) for d in datos]
    for que, d in pares:
        if not aparece(d, s):
            faltan.append(f"{que} en la sustitución" if que else f"tu dato {float(d):g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.spines.top': False, 'axes.spines.right': False})

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

URL_DATOS = "https://raw.githubusercontent.com/ggt13/DAQ_TSU/main/sesion_08_mantenimiento_predictivo_e_ia/datos/"

import warnings
import pandas as pd
from scipy.optimize import curve_fit

MAQUINAS_A = ["Bomba de agua helada del chiller 2", "Bomba de la torre de enfriamiento", "Bomba de agua de proceso",
              "Bomba de condensados", "Bomba del sistema contra incendio (prueba semanal)"]
U_A = 4.5                         # mm/s: límite entre las zonas C y D de ISO 10816-3 (sesión 7)
COLUMNAS = ["unidad", "ciclo", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
SENSORES = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s17", "s20", "s21"]   # 10 de los 14 que cambian con el desgaste
HORIZONTE = 30                    # «falla pronto»: le quedan 30 ciclos o menos
SUAVE = 10                        # promedio móvil de 10 ciclos (sesión 5)
C_INT_B = 50                      # miles de pesos por revisión planeada (parte B); la falla depende del código
COSTOS_FALLA_B = [100, 150, 200, 500, 750, 1000]
MOMENTOS = 10                     # momentos de decisión por motor de entrenamiento, elegidos al azar con semilla fija
M_MIN_B = 0.002                   # psia por ciclo: con una pendiente menor la recta no dice nada
UMBRALES_B = np.round(np.arange(47.60, 48.101, 0.05), 2)


def datos_s8(alumno):
    """Parte A: la bomba y su indicador de salud; la flota con su matriz de confusión y sus costos."""
    hi = parametro(alumno, "hi", 1.8, 3.6, 2)
    m = parametro(alumno, "m", 0.010, 0.060, 4)
    n = asignar(alumno, [30, 40, 50, 60], salto=1)
    p = int(round(n * parametro(alumno, "p", 0.15, 0.35, 3)))
    vp = int(round(p * parametro(alumno, "sens", 0.6, 0.95, 3)))
    fp = int(round((n - p) * parametro(alumno, "fpr", 0.05, 0.6, 3)))
    return dict(MAQUINA=asignar(alumno, MAQUINAS_A, salto=2), HI_HOY=hi, M_A=m, N_FLOTA=n,
                VP=vp, FN=p - vp, FP=fp, VN=n - p - fp,
                C_INT=int(round(parametro(alumno, "c_int", 15, 60, 0))),
                C_FALLA=int(round(parametro(alumno, "c_falla", 60, 250, 0))))


def clave_s8(alumno):
    d = datos_s8(alumno)
    return {"rul_A": (U_A - d["HI_HOY"]) / d["M_A"], "precision_A": d["VP"] / (d["VP"] + d["FP"]),
            "sens_A": d["VP"] / (d["VP"] + d["FN"]),
            "costo_pred_A": (d["VP"] + d["FP"]) * d["C_INT"] + d["FN"] * d["C_FALLA"]}


def _lee(nombre):
    """Lee un archivo de C-MAPSS del repositorio o, si se subió, de la carpeta de Colab."""
    errores = []
    for origen in (URL_DATOS + nombre, nombre):
        try:
            return pd.read_csv(origen, sep=r"\s+", header=None, compression="gzip")
        except Exception as e:
            errores.append(f"{origen}: {type(e).__name__} {e}"[:160])
    raise OSError(" | ".join(errores))


def carga_cmapss(alumno):
    """Entrenamiento, prueba y vida remanente real de la prueba. Si no se pueden leer, sintéticos marcados."""
    try:
        train, test = _lee("train_FD001.txt.gz"), _lee("test_FD001.txt.gz")
        rul = _lee("RUL_FD001.txt.gz")[0].to_numpy()
        train.columns = test.columns = COLUMNAS
        return train, test, rul, "C-MAPSS FD001, NASA Prognostics Center of Excellence"
    except Exception as e:
        print(f"⚠ No se pudieron leer los datos de C-MAPSS: {e}")
        print("⚠ Sube los tres archivos .txt.gz con el ícono de carpeta y vuelve a ejecutar. Mientras tanto, SINTÉTICOS.")
    g = np.random.default_rng(2027)
    filas_tr, filas_te, rul = [], [], []
    for u in range(1, 201):
        vida = int(g.integers(130, 360))
        t = np.arange(1, vida + 1)
        dano = np.exp(4 * (t / vida - 1))
        x = {f"s{i}": 100 + g.normal(0, 0.2, vida) for i in range(1, 22)}
        for k, s in enumerate(SENSORES):
            x[s] = 100 + (1 if k % 2 else -1) * 0.8 * dano + g.normal(0, 0.15, vida)
        x["s11"] = 47.3 + 0.8 * dano + g.normal(0, 0.15, vida)
        tabla = pd.DataFrame({"unidad": (u - 1) % 100 + 1, "ciclo": t, "op1": 0.0, "op2": 0.0, "op3": 100.0, **x})
        if u <= 100:
            filas_tr.append(tabla)
        else:
            corte = int(g.integers(int(0.4 * vida), vida - 5))
            filas_te.append(tabla.iloc[:corte])
            rul.append(vida - corte)
    return (pd.concat(filas_tr, ignore_index=True)[COLUMNAS], pd.concat(filas_te, ignore_index=True)[COLUMNAS],
            np.array(rul), "SINTÉTICOS (sin los datos de C-MAPSS)")


def suaviza(tabla):
    """Promedio móvil de 10 ciclos de cada sensor, motor por motor; HI es la s11 suavizada."""
    t = tabla.copy()
    t[SENSORES] = t.groupby("unidad")[SENSORES].transform(lambda x: x.rolling(SUAVE, min_periods=1).mean())
    t["HI"] = t["s11"]
    return t


def prepara_train(train):
    t = suaviza(train)
    t["vida"] = t.groupby("unidad").ciclo.transform("max")
    t["RUL"] = t.vida - t.ciclo
    return t


def umbral_falla(tr):
    """Umbral de falla del indicador: media − desviación estándar de su valor en el último ciclo de cada motor.

    Es un nivel conservador: lo alcanzan casi 9 de cada 10 motores antes de fallar."""
    fin = tr[tr.RUL == 0].HI
    return round(float(fin.mean() - fin.std()), 2)


def caso_b(alumno, tr):
    """Tu motor y tu «hoy»: entre el 72 y el 85 % de su vida, con tendencia clara en los últimos 30 ciclos.

    Sólo se asignan motores cuyo indicador sí llega al umbral de falla."""
    U = umbral_falla(tr)
    fin = tr[tr.RUL == 0]
    u = asignar(alumno, sorted(fin[fin.HI >= U].unidad))
    g = tr[tr.unidad == u]
    vida = int(g.vida.iloc[0])
    c = int(parametro(alumno, "frac_b", 0.72, 0.85, 3) * vida)
    while True:
        h = g[(g.ciclo <= c) & (g.ciclo > c - 30)]
        m, b = np.polyfit(h.ciclo, h.HI, 1)
        if m >= M_MIN_B or c >= vida - 10:
            break
        c += 5
    return dict(UNIDAD=int(u), VIDA=vida, CICLO_HOY=c, HI_HOY_B=round(float(m * c + b), 3), M_B=round(float(m), 5),
                RUL_REAL=vida - c)


def _expo(t, a, c, k):
    return a + c * np.exp(k * t / 100)


def rul_exponencial(g, c, U):
    """Ajusta HI = a + c·exp(k·t/100) a todo lo conocido hasta el ciclo c y calcula cuándo llega a U."""
    h = g[g.ciclo <= c]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            (a, cc, k), _ = curve_fit(_expo, h.ciclo, h.HI, p0=(47.3, 0.05, 1.5), maxfev=20000,
                                      bounds=([46, 1e-6, 1e-3], [48.5, 10, 10]))
        except Exception:
            return float("nan"), None
    if U <= a or cc <= 0 or k <= 0:
        return float("nan"), None
    return float(100 / k * np.log((U - a) / cc) - c), (a, cc, k)


def momentos(tr):
    """10 momentos de decisión al azar en la vida de cada motor de entrenamiento (1000 en total, semilla fija)."""
    g = np.random.default_rng(2027)
    idx = np.concatenate([g.choice(ix, MOMENTOS, replace=False) for _, ix in sorted(tr.groupby("unidad").groups.items())])
    return tr.loc[idx]


def costo(vp, fp, fn, c_falla):
    return (vp + fp) * C_INT_B + fn * c_falla


def tabla_umbrales(mom, c_falla):
    """Para cada umbral del HI: matriz de confusión de «falla en 30 ciclos o menos» en los momentos de decisión."""
    y = mom.RUL <= HORIZONTE
    filas = []
    for u in UMBRALES_B:
        p = mom.HI > u
        vp, fp, fn = int((p & y).sum()), int((p & ~y).sum()), int((~p & y).sum())
        filas.append({"umbral": u, "VP": vp, "FP": fp, "FN": fn,
                      "precision": vp / (vp + fp) if vp + fp else float("nan"), "sensibilidad": vp / (vp + fn),
                      "costo": costo(vp, fp, fn, c_falla)})
    return pd.DataFrame(filas)


def corte_ia(modelo, mom, c_falla):
    """El corte de probabilidad de la IA que menos cuesta en los momentos de entrenamiento: misma regla que el umbral."""
    y = (mom.RUL <= HORIZONTE).to_numpy()
    pr = modelo.predict_proba(mom[SENSORES])[:, 1]
    def c(q):
        m = matriz(pr > q, y)
        return costo(m["VP"], m["FP"], m["FN"], c_falla)
    return float(min(np.round(np.arange(0.05, 0.951, 0.05), 2), key=c))


def arbol_sin_limite(tr, ult, real):
    """Árbol de decisión sin límite de profundidad: sensibilidad en entrenamiento y en prueba (sobreajuste)."""
    from sklearn.tree import DecisionTreeClassifier
    y = tr.RUL <= HORIZONTE
    a = DecisionTreeClassifier(random_state=0).fit(tr[SENSORES], y)
    s_tr = matriz(a.predict(tr[SENSORES]), y)
    s_te = matriz(a.predict(ult[SENSORES]), real)
    return s_tr["VP"] / (s_tr["VP"] + s_tr["FN"]), s_te["VP"] / (s_te["VP"] + s_te["FN"])


def ultimo_ciclo(test):
    """El último ciclo registrado de cada motor de prueba, ya suavizado."""
    return suaviza(test).groupby("unidad").tail(1).reset_index(drop=True)


def matriz(pred, real):
    pred, real = np.asarray(pred, bool), np.asarray(real, bool)
    return dict(VP=int((pred & real).sum()), FP=int((pred & ~real).sum()), FN=int((~pred & real).sum()),
                VN=int((~pred & ~real).sum()))


def modelo_ia(tr):
    """Regresión logística con los 10 sensores suavizados: probabilidad de que el motor falle en 30 ciclos o menos."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
    m.fit(tr[SENSORES], tr.RUL <= HORIZONTE)
    return m


def revisa_umbral(u):
    if u is None:
        raise ValueError("Escribe UMBRAL_ALARMA, el umbral que elegiste de la tabla.")
    if not np.isclose(UMBRALES_B, float(u)).any():
        raise ValueError("Umbral de alarma: escribe uno de los valores de la tabla: "
                         + ", ".join(f"{v:.2f}" for v in UMBRALES_B) + ".")
    REGISTRO["Umbral de alarma"] = True
    print(f"✔ Umbral de alarma: {float(u):.2f} psia")

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S8"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Ejecuta la celda para ver **tu** bomba y **tu** flota. El **indicador de salud** de la bomba es su velocidad de
vibración RMS (sesión 7); la reparación debe hacerse antes de que llegue a **4.5 mm/s**, el límite de la zona D.

La flota tiene una **regla de alarma**. Cada bomba cae en una de cuatro casillas de la **matriz de confusión**:

| | Sí iba a fallar | No iba a fallar |
|---|---|---|
| **La alarma sonó** | VP: verdadero positivo | FP: falso positivo (revisión de más) |
| **No sonó** | FN: falso negativo (falla en operación) | VN: verdadero negativo |

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
rul_A  = 50.0
SUST_1 = "(4.5 − 3.0) mm/s / 0.03 mm/s por día = 50.0 días"
PRED_1 = "si el desgaste sigue igual la bomba llega al límite en unos cincuenta días"
```
Precisión y sensibilidad son fracciones entre 0 y 1 (no en %) y no tienen unidad: escribe «(adimensional)» o «sin
unidad» después del resultado.

In [ ]:
globals().update(datos_s8(alumno))
print(f"Tu bomba · {MAQUINA}")
print(f"  Indicador de salud hoy: HI_HOY = {HI_HOY} mm/s · tendencia de los últimos 30 días: M_A = {M_A} mm/s por día · "
      f"límite U_A = {U_A} mm/s")
print(f"\nTu flota de N_FLOTA = {N_FLOTA} bombas, un año con la regla de alarma:")
print(f"  VP = {VP} · FP = {FP} · FN = {FN} · VN = {VN}")
print(f"  Costo de una revisión planeada: C_INT = {C_INT} miles de pesos · de una falla en operación: C_FALLA = {C_FALLA} miles de pesos")

### ★ Ejercicio 1 · ¿Cuánto le queda a tu bomba? (3 pts)
**Para qué sirve:** es el número con el que se programa la reparación: refacciones, personal y paro planeado.

Si la tendencia sigue igual: **RUL = (U − HI_hoy) / m**. Calcula `rul_A` en días.

In [ ]:
# ✎ 1a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
rul_A = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, rul_A, SUST_1, PRED_1, datos={'tu indicador de hoy': HI_HOY, 'tu tendencia': M_A, 'el límite': U_A}, unidades=['día', 'dia', 'Día', 'Dia', 'DÍA'])

### Ejercicio 2 · ¿Cuántas alarmas eran ciertas? Precisión (2 pts)
**Para qué sirve:** si la precisión es baja, el personal deja de creerle a la alarma.

**Precisión = VP / (VP + FP)**: de todas las veces que sonó, la fracción que era una falla real.

In [ ]:
# ✎ 2a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
precision_A = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, precision_A, SUST_2, PRED_2, datos={'tu dato VP': VP, 'tu dato FP': FP}, unidades=['adimensional', 'Adimensional', 'sin unidad', 'Sin unidad'])

### Ejercicio 3 · ¿Cuántas fallas detectó? Sensibilidad (2 pts)
**Para qué sirve:** cada falla no detectada es un paro no planeado.

**Sensibilidad = VP / (VP + FN)**: de las bombas que sí iban a fallar, la fracción en que sonó la alarma.

In [ ]:
# ✎ 3a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
sens_A = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, sens_A, SUST_3, PRED_3, datos={'tu dato VP': VP, 'tu dato FN': FN}, unidades=['adimensional', 'Adimensional', 'sin unidad', 'Sin unidad'])

### Ejercicio 4 · ¿Cuánto costó el año con la alarma? (2 pts)
**Para qué sirve:** la decisión se toma en pesos: revisar de más cuesta poco, una falla en operación cuesta mucho.

Cada alarma (VP o FP) cuesta una revisión planeada; cada falla no detectada (FN) cuesta una falla en operación:
**costo = (VP + FP) · C_INT + FN · C_FALLA**. Resultado en miles de pesos.

In [ ]:
# ✎ 4a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
costo_pred_A = None   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, costo_pred_A, SUST_4, PRED_4, datos={'tu dato VP': VP, 'tu dato FP': FP, 'tu dato FN': FN, 'tu costo de revisión': C_INT, 'tu costo de falla': C_FALLA}, unidades=['miles de pesos', 'mil pesos', 'Miles de pesos', 'pesos', 'k$', 'MXN'])

In [ ]:
COSTO_CORR_A = (VP + FN) * C_FALLA
COSTO_PREV_A = N_FLOTA * C_INT                    # preventivo: revisar todas las bombas cada año
dias = np.arange(-30, int(rul_A) + 10)
fig, ax = plt.subplots()
ax.plot(dias[dias <= 0], HI_HOY + M_A * dias[dias <= 0] + alumno.rng("hi").normal(0, 0.04, (dias <= 0).sum()), "o",
        ms=3, color="#1F5FAD", label="indicador medido")
ax.plot(dias[dias >= 0], HI_HOY + M_A * dias[dias >= 0], "--", color="#D9822B", label=f"tendencia: llega en {rul_A:.0f} días")
ax.axhline(U_A, color="#C62828", lw=1, label=f"límite {U_A} mm/s"); ax.axvline(0, color="#888", lw=0.8)
ax.set_xlabel("días desde hoy"); ax.set_ylabel("velocidad RMS (mm/s)"); ax.legend(fontsize=8)
titulo(ax, "Vida remanente de tu bomba", alumno)
firma(fig, alumno); plt.show()
print(f"Mantenimiento correctivo (dejar fallar las {VP + FN} bombas que fallan): {COSTO_CORR_A} miles de pesos")
print(f"Mantenimiento preventivo (revisar las {N_FLOTA} bombas, suponiendo que la revisión evita todas las fallas): "
      f"{COSTO_PREV_A} miles de pesos")
print(f"Mantenimiento predictivo con la alarma: {costo_pred_A:.0f} miles de pesos")

### Cierre (1 pt)
¿Conviene la alarma frente a dejar fallar las bombas? Compara los costos impresos arriba (también el del preventivo)
y di qué importa más en tu flota, la precisión o la sensibilidad. Al menos 20 palabras, con los costos del
predictivo y del correctivo.

In [ ]:
CIERRE = ""   # ✎ escribe aquí

revisa_texto('Cierre', CIERRE, {'el costo con predictivo': costo_pred_A,
             'el costo correctivo': COSTO_CORR_A})

## Parte B · Práctica (80 min)
**Objetivo:** con datos de degradación hasta la falla, ver cuándo funciona una predicción sencilla y cuándo hace falta
un modelo, y medir los dos en máquinas nuevas.

**C-MAPSS FD001** (NASA): 100 motores de entrenamiento que operan desde nuevos hasta fallar y 100 de prueba cuyo
registro termina antes de la falla (se sabe cuánto les quedaba). Un ciclo es un vuelo. El indicador de salud es el
sensor **s11**, la presión a la salida del compresor de alta (psia), suavizado con un promedio móvil de 10 ciclos
(sesión 5). El **umbral de falla** `U_B` es el promedio del indicador en el último ciclo de los 100 motores menos una
desviación estándar: un nivel conservador que casi 9 de cada 10 motores alcanzan antes de fallar. Son datos simulados con un modelo físico del motor y ruido de sensor; se usan en todo el mundo para
probar métodos de pronóstico.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Datos y tu motor | 10 | Ves la flota y tu motor hasta tu «hoy» |
| B2 · Vida remanente con una recta | 15 | Calculas a mano cuántos ciclos le quedan y lo comparas con lo real |
| B3 · Umbral por costo | 15 | Eliges el umbral de alarma que menos cuesta en el entrenamiento, con tu costo de falla |
| B4 · Umbral contra IA en motores nuevos | 20 | Matriz de confusión de tu umbral y de un modelo de IA en los 100 motores de prueba; un modelo que memoriza |
| B5 · Conclusión | 10 | Con tus números, y qué hace falta para usar IA en planta |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
if "TRAIN" not in globals() or FUENTE.startswith("SINTÉTICOS"):
    TRAIN, TEST, RUL_TEST, FUENTE = carga_cmapss(alumno)
TR = prepara_train(TRAIN)
U_B = umbral_falla(TR)
globals().update(caso_b(alumno, TR))
G = TR[TR.unidad == UNIDAD]
print(f"B1 · Fuente: {FUENTE}")
print(f"     Entrenamiento: {TR.unidad.nunique()} motores, vida de {TR.vida.min()} a {TR.vida.max()} ciclos · "
      f"umbral de falla del indicador: U_B = {U_B} psia (último ciclo de la flota: media − desviación)")
C_FALLA_B = asignar(alumno, COSTOS_FALLA_B, salto=3)
print(f"     Tu motor: {UNIDAD} · tu «hoy»: ciclo {CICLO_HOY} · tu costo de una falla en operación: C_FALLA_B = "
      f"{C_FALLA_B} miles de pesos (revisión planeada: {C_INT_B})")
fig, ax = plt.subplots(figsize=(10, 4))
for u, g in TR.groupby("unidad"):
    ax.plot(g.ciclo, g.HI, color="#DDDDDD", lw=0.5)
ax.plot(G.ciclo[G.ciclo <= CICLO_HOY], G.HI[G.ciclo <= CICLO_HOY], color="#1F5FAD", lw=2, label=f"tu motor {UNIDAD} hasta hoy")
ax.axhline(U_B, color="#C62828", lw=1, label=f"umbral de falla {U_B}")
ax.set_xlabel("ciclo"); ax.set_ylabel("s11 suavizada (psia)"); ax.legend(fontsize=8)
titulo(ax, "Indicador de salud de la flota", alumno)
firma(fig, alumno); plt.show()

### B2 · Vida remanente con una recta
El cuaderno ajusta una recta a los últimos 30 ciclos de tu motor (`M_B`, en psia por ciclo) y da el valor de hoy en
esa recta (`HI_HOY_B`). Con la fórmula del ★ calcula `RUL_PRED_B`, en ciclos. Después verás cuántos le quedaban
de verdad y qué predice una curva exponencial ajustada a toda su historia.

In [ ]:
print(f"HI_HOY_B = {HI_HOY_B:.3f} psia · M_B = {M_B:.5f} psia por ciclo · U_B = {U_B:.2f} psia")

In [ ]:
RUL_PRED_B = None   # ✎ tu vida remanente predicha, en ciclos

completa(RUL_PRED_B, "tu vida remanente predicha")
RUL_EXP, par = rul_exponencial(G, CICLO_HOY, U_B)
t_fut = np.arange(CICLO_HOY - 30, max(VIDA, CICLO_HOY + min(RUL_PRED_B, 2 * VIDA)) + 10)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(G.ciclo[G.ciclo <= CICLO_HOY], G.HI[G.ciclo <= CICLO_HOY], color="#1F5FAD", lw=1.5, label="lo que se sabía hoy")
ax.plot(G.ciclo[G.ciclo > CICLO_HOY], G.HI[G.ciclo > CICLO_HOY], color="#9DB7D5", lw=1.5, label="lo que pasó después")
ax.plot(t_fut, HI_HOY_B + M_B * (t_fut - CICLO_HOY), "--", color="#D9822B", label=f"recta: {RUL_PRED_B:.0f} ciclos")
if par is not None:
    ax.plot(t_fut, par[0] + par[1] * np.exp(par[2] * t_fut / 100), ":", color="#6A1B9A", lw=2,
            label=f"exponencial: {RUL_EXP:.0f} ciclos")
ax.axhline(U_B, color="#C62828", lw=1); ax.axvline(CICLO_HOY, color="#888", lw=0.8)
ax.axvline(VIDA, color="#C62828", ls=":", lw=1, label=f"falla real: {RUL_REAL} ciclos después de hoy")
ax.set_ylim(G.HI.min() - 0.05, U_B + 0.25); ax.set_xlabel("ciclo"); ax.set_ylabel("s11 suavizada (psia)")
ax.legend(fontsize=8, loc="upper left"); titulo(ax, f"Vida remanente del motor {UNIDAD}", alumno)
firma(fig, alumno); plt.show()
print(f"Recta: {RUL_PRED_B:.0f} ciclos · exponencial: " + ("no se pudo ajustar" if np.isnan(RUL_EXP) else f"{RUL_EXP:.0f} ciclos")
      + f" · real: {RUL_REAL} ciclos")
print("El desgaste se acelera: la recta supone que sigue al ritmo de hoy. En 9 de cada 10 casos de esta flota promete "
      "más vida de la que hay; la exponencial queda más cerca en casi 8 de cada 10.")

### B3 · Umbral de alarma por costo
La regla: **alarma si el indicador pasa un umbral**. Para elegirlo se simulan 1000 decisiones: 10 momentos al azar
en la vida de cada motor de entrenamiento, en los que se sabe si al motor le quedaban 30 ciclos o menos. La tabla da,
para cada umbral, la matriz de confusión, la precisión, la sensibilidad y el costo con **tu** costo de falla. Un umbral
bajo da muchas alarmas falsas (aunque muchas son **tempranas**: motores con 31 a 50 ciclos); uno alto deja pasar
fallas. **Elige el de menor costo** y escríbelo en `UMBRAL_ALARMA`.

In [ ]:
MOM = momentos(TR)
TABLA = tabla_umbrales(MOM, C_FALLA_B)
print(f"Umbral en psia · VP, FP y FN en 1000 decisiones · costo en miles de pesos con revisión {C_INT_B} y falla {C_FALLA_B}")
print(TABLA.to_string(index=False, formatters={"umbral": "{:.2f}".format, "precision": "{:.2f}".format,
                                               "sensibilidad": "{:.2f}".format, "costo": "{:,.0f}".format}))

In [ ]:
UMBRAL_ALARMA = None   # ✎ el umbral que elegiste, en psia

revisa_umbral(UMBRAL_ALARMA)
UMBRAL_ALARMA = float(UMBRAL_ALARMA)

### B4 · Tu umbral contra un modelo de IA, en 100 motores que no se usaron para ajustar nada
El modelo de **aprendizaje automático** (regresión logística) aprendió del entrenamiento a combinar **10 sensores**,
no sólo s11, y da una probabilidad de falla; su corte de alarma se eligió con **tu mismo costo** y las mismas 1000
decisiones que tu umbral, para que la comparación sea justa. Los dos se prueban en el último ciclo de los 100 motores
de prueba: ¿les quedaban 30 ciclos o menos? Sólo 25 de ellos fallaban pronto, así que **un motor cambia la
sensibilidad en 0.04**: diferencias de ese tamaño no distinguen los dos métodos. Con la matriz de **tu umbral**
calcula la sensibilidad en prueba, `SENS_PRUEBA` (entre 0 y 1).

Al final, un **árbol de decisión sin límite** muestra el riesgo de la IA: memoriza el entrenamiento y le va peor en
motores que no vio (**sobreajuste**). Por eso un modelo se evalúa siempre en máquinas **distintas** de las que usó
para aprender; si se mezclan ciclos del mismo motor en entrenamiento y prueba, el resultado sale engañosamente bueno
(**fuga de datos**).

In [ ]:
ULT = ultimo_ciclo(TEST)
REAL_PRUEBA = RUL_TEST <= HORIZONTE
M_UMB = matriz(ULT.HI > UMBRAL_ALARMA, REAL_PRUEBA)
IA = modelo_ia(TR)
Q_IA = corte_ia(IA, MOM, C_FALLA_B)
M_IA = matriz(IA.predict_proba(ULT[SENSORES])[:, 1] > Q_IA, REAL_PRUEBA)
SENS_IA = M_IA["VP"] / (M_IA["VP"] + M_IA["FN"])
PREC_IA = M_IA["VP"] / max(1, M_IA["VP"] + M_IA["FP"])
print(f"100 motores de prueba · {int(REAL_PRUEBA.sum())} de ellos fallaban en 30 ciclos o menos")
print(f"Tu umbral ({UMBRAL_ALARMA:.2f} psia):  VP = {M_UMB['VP']} · FP = {M_UMB['FP']} · FN = {M_UMB['FN']} · VN = {M_UMB['VN']}")
print(f"Modelo de IA (10 sensores, corte {Q_IA:.2f}): VP = {M_IA['VP']} · FP = {M_IA['FP']} · FN = {M_IA['FN']} · "
      f"VN = {M_IA['VN']} · sensibilidad {SENS_IA:.2f} · precisión {PREC_IA:.2f}")

In [ ]:
SENS_PRUEBA = None   # ✎ la sensibilidad de tu umbral en prueba

completa(SENS_PRUEBA, "la sensibilidad de tu umbral")
if not 0 <= SENS_PRUEBA <= 1:
    raise ValueError("La sensibilidad es una fracción entre 0 y 1, no un porcentaje: VP / (VP + FN).")
PREC_UMB = M_UMB["VP"] / max(1, M_UMB["VP"] + M_UMB["FP"])
COSTO_UMB = costo(M_UMB["VP"], M_UMB["FP"], M_UMB["FN"], C_FALLA_B)
COSTO_IA = costo(M_IA["VP"], M_IA["FP"], M_IA["FN"], C_FALLA_B)
S_ARB_TR, S_ARB_TE = arbol_sin_limite(TR, ULT, REAL_PRUEBA)
fila = TABLA[np.isclose(TABLA.umbral, UMBRAL_ALARMA)].iloc[0]
RESULTADOS = {"rul_A": rul_A, "precision_A": precision_A, "sens_A": sens_A, "costo_pred_A": costo_pred_A,
              "unidad_B": UNIDAD, "ciclo_hoy": CICLO_HOY, "u_B": U_B, "rul_pred_B": RUL_PRED_B,
              "rul_exp_B": -1.0 if np.isnan(RUL_EXP) else RUL_EXP, "rul_real_B": RUL_REAL, "c_falla_B": C_FALLA_B,
              "umbral_alarma": UMBRAL_ALARMA,
              "sens_entrena": float(fila.sensibilidad), "sens_prueba": SENS_PRUEBA, "prec_prueba": PREC_UMB,
              "sens_ia": SENS_IA, "prec_ia": PREC_IA, "costo_umbral": COSTO_UMB, "costo_ia": COSTO_IA}
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, M, nom in ((axs[0], M_UMB, f"Tu umbral {UMBRAL_ALARMA:.2f}"), (axs[1], M_IA, "Modelo de IA")):
    mat = np.array([[M["VP"], M["FP"]], [M["FN"], M["VN"]]])
    ax.imshow(mat, cmap="Blues")
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=14, color="white" if v > mat.max() / 2 else "black")
    ax.set_xticks([0, 1], ["fallaba pronto", "no"]); ax.set_yticks([0, 1], ["alarma", "sin alarma"]); ax.set_title(nom)
fig.suptitle(f"Matrices de confusión en 100 motores de prueba · {alumno.nombre}")
firma(fig, alumno, RESULTADOS); plt.tight_layout(rect=(0, 0.06, 1, 0.95)); plt.show()
print(f"Sensibilidad de tu umbral · en las decisiones de entrenamiento {fila.sensibilidad:.2f} · en prueba {SENS_PRUEBA:.2f}")
print(f"Costo en prueba · tu umbral {COSTO_UMB:,} · modelo de IA {COSTO_IA:,} miles de pesos")
print(f"Árbol de decisión sin límite (memoriza): sensibilidad {S_ARB_TR:.2f} en entrenamiento y {S_ARB_TE:.2f} en prueba")

### B5 · Conclusión
¿Qué tan cerca quedó la recta de la vida real de tu motor, y por qué? ¿Tu umbral funcionó igual en los motores
nuevos que en el entrenamiento? ¿El modelo de IA fue mejor, igual o peor que tu umbral en prueba, y qué haría falta
para usarlo en planta? Al menos 25 palabras, con tu vida remanente predicha, la real y las dos sensibilidades de
prueba.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'tu vida remanente predicha': RUL_PRED_B, 'la vida remanente real': RUL_REAL,
              'la sensibilidad de tu umbral en prueba': SENS_PRUEBA,
              'la sensibilidad del modelo de IA': SENS_IA}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S8_<código>.csv` con la historia de tu motor: lo que se sabía hoy, lo que pasó después y la recta.
En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar → Subir** → elige el archivo → **Reemplazar
la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
ciclos = np.arange(1, VIDA + 1)
hi = G.set_index("ciclo").HI.reindex(ciclos).to_numpy()
recta = HI_HOY_B + M_B * (ciclos - CICLO_HOY)
ruta = f"S8_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"ciclo": ciclos,
              "HI_hasta_hoy": [round(float(v), 3) if c <= CICLO_HOY else "" for c, v in zip(ciclos, hi)],
              "HI_despues": [round(float(v), 3) if c > CICLO_HOY else "" for c, v in zip(ciclos, hi)],
              "recta": [round(float(v), 3) if c >= CICLO_HOY - 30 and v <= U_B + 0.1 else "" for c, v in zip(ciclos, recta)]},
             extra={"fuente": FUENTE, "motor": f"{UNIDAD}, hoy en el ciclo {CICLO_HOY}", "maquina": MAQUINA})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Del curso a tu proyecto integrador
Las ocho sesiones forman una cadena que puedes aplicar a una variable de tu trabajo:
**adquirir** con el intervalo y la resolución correctos (S1–S2) → **calibrar** y conocer la **incertidumbre** (S3–S4)
→ **filtrar** y ver la **tendencia** (S5) → **pronosticar** (S6) → saber **qué** falla con el espectro (S7) → decidir
**cuándo** intervenir con un **indicador de salud**, un **umbral** elegido por costo y su **matriz de confusión** (S8).
En tu proyecto: elige el indicador de salud de tu variable, su umbral y qué costaría cada tipo de error.

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")